# Antarctic Vision
## Detecting Penguins, Icebergs, and Other Antarctic Features with YOLOv5

The main purpose of this project was to be able to identify three main features - Penguins, Icebergs, and Rocks, in an antarctic themed picture, regardless of their size or orientation. Utilising YOLOv5, the output image will display selection squares around each identified feature.

Additionally, the image will showcase the segmentation of these features, which is essentially a process that involves highlighting each individual feature in the output image. This is a complex task especially when dealing with multiple features in a single image.


The initial python script checks the current working directory to ensure that environment setup is performed correctly. This maintains consistency in the working environment for all users.

In [ ]:
# Ensure the user is in the correct directory to set up the required environment

# Check current directory
import os
current_directory = os.getcwd()
print("Current directory:", current_directory)

# If current directory is not /content, change it
if current_directory != '/content':
    os.chdir('/content')
    print("Changed directory to /content")

The next code snippet clones the yolov5 Github Repository into the google colab environment. It installs all the required dependencies and modules to ensure the environment is prepared to use the full capabilities of the yolov5 vision AI.

In [ ]:
!git clone https://github.com/ultralytics/yolov5  # clone yolov5 github repository into colab environment

# # change directory
%cd yolov5

# install required dependencies
%pip install -qr requirements.txt comet_ml

# install roboflow
%pip install -q roboflow

# install opencv
%pip install opencv-python-headless

# install required libraries
import torch
import utils
import argparse
import csv
import platform
import sys
from IPython.display import Image, clear_output

display = utils.notebook_init()  # initiliaze display for notebook
print(f"Setup complete. Using torch {torch.__version__} ({torch.cuda.get_device_properties(0).name if torch.cuda.is_available() else 'CPU'})")

The next code snippet downloads some sample images from a google drive folder created for an initial demonstration of the capabilites of the default yolov5 model. The next step will be to train the model on custom datasets designed specifically to target arctic landscapes for object detection and instance segmentation.

In [ ]:
!pip install gdown

#collapse-output
import gdown
url = "https://drive.google.com/drive/folders/1LuItduSt6dN89uFuknqy6nGsb1Y7Ib9F?usp=sharing"
gdown.download_folder(url, quiet=True, use_cookies=False)

The following code performs object detection and instance segmentation using the yolov5 default model on three sample penguin images.

In [ ]:
from IPython.display import Image, display

# Run the detection and save the results to specified folder location
!python detect.py --weights yolov5x6.pt --source "/content/yolov5/penguin_sample_set/penguin_generic.jpg"  --exist-ok
!python detect.py --weights yolov5x6.pt --source "/content/yolov5/penguin_sample_set/penguin_group.jpeg" --exist-ok
!python segment/predict.py --weights yolov5m-seg.pt --source "/content/yolov5/penguin_sample_set/emperor_penguins.jpg" --exist-ok

# Display the results
print("\n")
print("Object Detection Ex. #1")
display(Image(filename='/content/yolov5/runs/detect/exp/penguin_generic.jpg', width=600))
print("\n")
print("Object Detection Ex. #2")
display(Image(filename='/content/yolov5/runs/detect/exp/penguin_group.jpeg', width=600))
print("\n")
print("Instance Segmentation Ex. #1")
display(Image(filename='/content/yolov5/runs/predict-seg/exp/emperor_penguins.jpg', width=600))

This next code snippet further demonstrates the capabilities of the yolov5 model as it performs object detection on a video of penguins walking as a group. This script detects penguins in each frame of the video, and draws bounding boxes around them.

*Disclosure: The execution of this code takes approximatley 5 - 7 minutes depending on the user's system.*

In [ ]:
import itertools
import math
import warnings

!pip install -q mediapy
import mediapy as media
import matplotlib.pyplot as plt
import numpy as np

!python detect.py --weights yolov5x6.pt --source "/content/yolov5/penguin_sample_set/penguin_demo_video.mp4" --exist-ok

print("\nObject Detection Ex. #3")
url = "/content/yolov5/runs/detect/exp/penguin_demo_video.mp4"
media.show_video(media.read_video(url), fps=25, height=500)

The following code implements the Roboflow API to import three custom datasets to train the yolov5 AI Vision model to detect arctic features, specifically penguins, rocks, and icebergs. This will train the default model to adapt to our specific project objectives. The first dataset containing penguin images was created using a google API that implements Open Images Dataset V7 and entensions. These images were then imported into Roboflow for formatting, annotation, and pre-processing.

The other datasets contain rock instance sementation data generated by Oregon State University and iceberg instance segmentation data generated by an independent research conducting ocean science research in France. Both of these datasets were processed via Roboflow, which is data processing pipeline that allows users to organize, annotate, and format large datasets for machine learning tasks.

These datasets will be used to train the yolov5 AI vision model to optimize it for our project's objectives.

In [ ]:
!pip install roboflow

import os
from roboflow import Roboflow

# Retreive Penguin Object Detection Data Set (Custom dataset developed by our project team
# using roboflow and Open Images Dataset V7: https://app.roboflow.com/benjaminphilipenkodev/arctic-landscape-object-detect/deploy)
rf = Roboflow(api_key=os.environ["ROBOFLOW_API_KEY"])
project = rf.workspace("benjaminphilipenkodev").project("arctic-landscape-object-detect")
version = project.version(1)
penguin_dataset = version.download("yolov5", location="/content/datasets/penguin_images")


# Retreive Rock Instance Segmentation Data Set (Roboflow opensource API: https://universe.roboflow.com/oregon-state-university-ny6yy/rock-dataset)
rf = Roboflow(api_key=os.environ["ROBOFLOW_API_KEY"])
project = rf.workspace("oregon-state-university-ny6yy").project("rock-dataset")
version = project.version(3)
rock_dataset = version.download( "yolov5", location="/content/datasets/rock_images")

# Retreive Iceberg Instance Segmentation Data Set (Roboflow opensource API: https://universe.roboflow.com/locean/locean/dataset/5)
rf = Roboflow(api_key=os.environ["ROBOFLOW_API_KEY"])
project = rf.workspace("locean").project("locean")
version = project.version(5)
iceberg_dataset = version.download("yolov5", location="/content/datasets/iceberg_images")


The following segment allows for users to adjust the training parameters. These parameters play a crucial role in defining the overall performance of the model. For context, the first time the model was trained the parameters were reduced to img 416, batch 16, and epoch 1. This proved to be a poorly trained model. The parameters for the next model in our emprical experimental process were set to img 416, batch 16, and epoch 1. This improved the overall performance of the model greatly.

In [ ]:
!python train.py --img 640 --batch 16 --epochs 150 --data {penguin_dataset.location}/data.yaml --weights yolov5x6.pt --cache

The TensorBoard allows for users to analyze the current models overall perfomance with generated PR-Curves.

In [ ]:
# Start tensorboard
# Launch after you have started training
# logs save in the folder "runs"
%load_ext tensorboard
%tensorboard --logdir runs

The following code allows users to specify the file path to the trained model and input images to the model. Confidence thresholds can also be set, allows users to control what what detections are valid. i.e. if conf is set to 0.1, then only detections with a confidence score above 0.1 will be displayed.

In [ ]:
!python detect.py --weights runs/train/exp/weights/best.pt --img 640 --conf 0.1 --source {penguin_dataset.location}/test/images

This allows for users to visualize multiple images within a directory.

In [ ]:
#display inference on ALL test images

import glob
from IPython.display import Image, display

for imageName in glob.glob('/content/yolov5/runs/detect/exp2/*.jpg'): #assuming JPG
    display(Image(filename=imageName))
    print("\n")

In [ ]:
#export your model's weights for future use
from google.colab import files
files.download('./runs/train/exp/weights/best.pt')